# 04 Time Series Forecasting

Build a daily temperature series, train forecasting models, compare metrics, and save model artifacts.


In [1]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = next(
    (
        candidate
        for candidate in (NOTEBOOK_DIR, *NOTEBOOK_DIR.parents)
        if (candidate / "src").exists() and (candidate / "data").exists()
    ),
    NOTEBOOK_DIR,
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "GlobalWeatherRepository.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


In [2]:
import joblib
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from statsmodels.tsa.arima.model import ARIMA

from src.eval import evaluate_forecast, summarize_results
from src.features import build_daily_temperature_series
from src.preprocessing import load_processed_data
from src.visualize import save_figure


In [3]:
anomaly_free_path = PROCESSED_DIR / "weather_without_anomalies.csv"
clean_path = PROCESSED_DIR / "clean_weather_data.csv"
source_path = anomaly_free_path if anomaly_free_path.exists() else clean_path

df = load_processed_data(source_path)
ts = build_daily_temperature_series(df)

print(f"Loaded source data from: {source_path}")
print(f"Daily series length: {len(ts)}")
print(f"Series range: {ts.index.min()} to {ts.index.max()}")
display(ts.head())


Loaded source data from: d:\Projects 2025\weather-forecasting-eda-ml\data\processed\weather_without_anomalies.csv
Daily series length: 708
Series range: 2024-05-16 00:00:00 to 2026-04-24 00:00:00


last_updated
2024-05-16    23.773481
2024-05-17    24.419429
2024-05-18    25.416583
2024-05-19    25.365775
2024-05-20    25.539674
Name: temperature_celsius, dtype: float64

## 1b  Statistical Analysis of the Series

Before fitting any model we check stationarity (ADF test) and inspect autocorrelation structure (ACF / PACF) to guide model order selection.

In [ ]:
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Attach explicit daily frequency and forward-fill any gaps
ts_daily = ts.asfreq("D").ffill()
train_size_d = int(len(ts_daily) * 0.8)
train_daily  = ts_daily.iloc[:train_size_d]
test_daily   = ts_daily.iloc[train_size_d:]

print(f"Daily series: {len(ts_daily)} observations  |  "
      f"train={len(train_daily)}  test={len(test_daily)}")

# ── Augmented Dickey-Fuller stationarity test ─────────────────────────────
adf_stat, adf_p, _, _, adf_crit, _ = adfuller(ts_daily, autolag="AIC")
print(f"\nADF Statistic : {adf_stat:.4f}")
print(f"p-value       : {adf_p:.6f}")
print(f"Critical 5%   : {adf_crit['5%']:.4f}")
print(f"Verdict       : {'STATIONARY ✓' if adf_p < 0.05 else 'NON-STATIONARY — differencing needed'}")

# ── ACF / PACF ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
plot_acf( ts_daily, lags=40, ax=axes[0], title="ACF  — Daily Global Avg Temperature")
plot_pacf(ts_daily, lags=40, ax=axes[1], title="PACF — Daily Global Avg Temperature")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "acf_pacf.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved reports/figures/acf_pacf.png")

In [4]:
if len(ts) < 60:
    raise ValueError("The daily time series is too short for the planned forecasting workflow.")

train_size = int(len(ts) * 0.8)
train = ts.iloc[:train_size]
test = ts.iloc[train_size:]

arima_model = ARIMA(train, order=(5, 1, 2))
arima_fit = arima_model.fit()
arima_forecast = pd.Series(arima_fit.forecast(steps=len(test)).to_numpy(), index=test.index)

forecast_results = {"arima": evaluate_forecast(test, arima_forecast)}


D:\Projects 2025\weather-forecasting-eda-ml\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
D:\Projects 2025\weather-forecasting-eda-ml\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
D:\Projects 2025\weather-forecasting-eda-ml\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


D:\Projects 2025\weather-forecasting-eda-ml\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
D:\Projects 2025\weather-forecasting-eda-ml\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(


### SARIMA — Seasonal ARIMA

SARIMA adds seasonal AR and MA terms. With `seasonal_order=(1,0,1,7)` we capture weekly periodicity without extra differencing, which plain ARIMA(5,1,2) misses entirely.

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

# SARIMA(1,1,1)(1,0,1,7) — base ARIMA + weekly seasonal AR and MA terms
sarima_fit = SARIMAX(
    train_daily,
    order=(1, 1, 1),
    seasonal_order=(1, 0, 1, 7),
    enforce_stationarity=False,
    enforce_invertibility=False,
).fit(disp=False)

sarima_pred    = sarima_fit.forecast(steps=len(test_daily))
sarima_forecast = pd.Series(sarima_pred.values, index=test_daily.index)

forecast_results["sarima"] = evaluate_forecast(test_daily, sarima_forecast)
print(f"SARIMA AIC : {sarima_fit.aic:.2f}")
print("SARIMA Metrics:")
for k, v in forecast_results["sarima"].items():
    print(f"  {k:6s}: {v:.4f}")

In [5]:
prophet_model = None
prophet_forecast = None

try:
    from prophet import Prophet

    prophet_train = train.reset_index()
    prophet_train.columns = ["ds", "y"]

    prophet_model = Prophet()
    prophet_model.fit(prophet_train)

    future = prophet_model.make_future_dataframe(periods=len(test), freq="D")
    forecast_frame = prophet_model.predict(future)
    prophet_forecast = forecast_frame.set_index("ds").loc[test.index, "yhat"]
    forecast_results["prophet"] = evaluate_forecast(test, prophet_forecast)
except Exception as exc:
    print(f"Prophet training skipped: {exc}")


23:44:04 - cmdstanpy - INFO - Chain [1] start processing


23:44:05 - cmdstanpy - INFO - Chain [1] done processing


### XGBoost — ML-based Time-Series Forecasting

Convert the series into a supervised regression problem using lag values (1–30 days), rolling means, and cyclical date encodings (day-of-year sin/cos for annual seasonality, day-of-week for weekly cycles). XGBoost is then trained on the resulting feature matrix.

In [ ]:
import numpy as np
from xgboost import XGBRegressor


def build_lag_features(series: pd.Series) -> pd.DataFrame:
    """Create supervised learning features from a time series."""
    doy = series.index.dayofyear
    dow = series.index.dayofweek
    frame = pd.DataFrame(index=series.index)
    for lag in [1, 2, 3, 7, 14, 21, 30]:
        frame[f"lag_{lag}"] = series.shift(lag)
    frame["roll_7"]  = series.shift(1).rolling(7).mean()
    frame["roll_14"] = series.shift(1).rolling(14).mean()
    frame["roll_30"] = series.shift(1).rolling(30).mean()
    frame["doy_sin"] = np.sin(2 * np.pi * doy / 365.25)
    frame["doy_cos"] = np.cos(2 * np.pi * doy / 365.25)
    frame["dow_sin"] = np.sin(2 * np.pi * dow / 7)
    frame["dow_cos"] = np.cos(2 * np.pi * dow / 7)
    frame["target"]  = series.values
    return frame.dropna()


feat_df   = build_lag_features(ts_daily)
feat_cols = [c for c in feat_df.columns if c != "target"]
cutoff    = train_daily.index[-1]
train_f   = feat_df[feat_df.index <= cutoff]
test_f    = feat_df[feat_df.index >  cutoff]

xgb = XGBRegressor(
    n_estimators=500, learning_rate=0.03, max_depth=5,
    subsample=0.8, colsample_bytree=0.8, random_state=42,
)
xgb.fit(train_f[feat_cols], train_f["target"], verbose=False)
xgb_forecast = pd.Series(xgb.predict(test_f[feat_cols]), index=test_f.index)

forecast_results["xgboost_lag"] = evaluate_forecast(test_f["target"], xgb_forecast)
print("XGBoost + Lag Features Metrics:")
for k, v in forecast_results["xgboost_lag"].items():
    print(f"  {k:6s}: {v:.4f}")

# Feature importance
imp_df = (
    pd.DataFrame({"feature": feat_cols, "importance": xgb.feature_importances_})
    .sort_values("importance", ascending=False)
    .head(10)
)
print("\nTop 10 Feature Importances:")
print(imp_df.to_string(index=False))

In [ ]:
# Align ARIMA forecast to test index (integer-indexed due to missing freq info)
arima_forecast = pd.Series(arima_forecast.values, index=test.index)

results_df = summarize_results(forecast_results)
display(results_df)

# ── Figure 1: all forecasts vs actuals ───────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 6))
train_daily[-90:].plot(ax=ax, label="Train (last 90 d)", color="steelblue", linewidth=2)
test_daily.plot(ax=ax, label="Actual", color="black", linewidth=2)

model_styles = {
    "arima":        ("ARIMA",          "orange",    arima_forecast),
    "sarima":       ("SARIMA",         "purple",    sarima_forecast),
    "xgboost_lag":  ("XGBoost + Lag",  "crimson",   xgb_forecast),
}
if prophet_forecast is not None:
    model_styles["prophet"] = ("Prophet", "limegreen", prophet_forecast.reindex(test_daily.index))

for key, (label, color, fc) in model_styles.items():
    r2  = forecast_results[key]["r2"]
    mae = forecast_results[key]["mae"]
    fc.plot(ax=ax, label=f"{label}  R²={r2:.3f}  MAE={mae:.2f}°C",
            linestyle="--", color=color, linewidth=1.4)

ax.set_title("Daily Temperature Forecast — Model Comparison", fontsize=14, fontweight="bold")
ax.legend(ncol=2, fontsize=9)
ax.set_ylabel("Temperature (°C)")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "forecast_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Figure 2: R² and MAE bar charts ──────────────────────────────────────
model_names = list(forecast_results.keys())
r2_vals  = [forecast_results[m]["r2"]  for m in model_names]
mae_vals = [forecast_results[m]["mae"] for m in model_names]
colors   = ["#4f86c6", "#9c5fc4", "#e84c4c", "#4caf7d", "#e07b3a"][:len(model_names)]

fig2, axes2 = plt.subplots(1, 2, figsize=(13, 5))
for ax2, vals, title, ylabel in zip(
    axes2,
    [r2_vals, mae_vals],
    ["R² — Higher is Better", "MAE (°C) — Lower is Better"],
    ["R²", "MAE (°C)"],
):
    bars = ax2.bar(model_names, vals, color=colors, edgecolor="white")
    ax2.axhline(0, color="black", linewidth=0.8, linestyle="--")
    ax2.set_title(title, fontsize=12, fontweight="bold")
    ax2.set_ylabel(ylabel)
    for bar, v in zip(bars, vals):
        ypos = max(v, 0) + abs(max(vals) - min(vals)) * 0.02
        ax2.text(bar.get_x() + bar.get_width() / 2, ypos,
                 f"{v:.3f}", ha="center", va="bottom", fontsize=9, fontweight="bold")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "forecast_r2_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figures saved.")

In [ ]:
forecast_metrics_path = PROJECT_ROOT / "reports" / "forecast_metrics.csv"
results_df.to_csv(forecast_metrics_path)
print(f"Saved forecast metrics   → {forecast_metrics_path}")

joblib.dump(arima_fit,   MODELS_DIR / "arima_temperature_model.joblib")
joblib.dump(sarima_fit,  MODELS_DIR / "sarima_temperature_model.joblib")
joblib.dump(xgb,         MODELS_DIR / "xgboost_lag_temperature_model.joblib")
print("Saved ARIMA, SARIMA, XGBoost models → models/")

if prophet_model is not None:
    joblib.dump(prophet_model, MODELS_DIR / "prophet_temperature_model.joblib")
    print("Saved Prophet model → models/")

print("\n── Final Rankings (sorted by RMSE) ──")
print(results_df.to_string())